# Live Modidx Cache
> Cached _modidx tree with usage tallies and persistence

In [ ]:
#| default_exp tasks.cache

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Set, ClassVar
from dataclasses import dataclass, field, asdict
from pathlib import Path
from collections import defaultdict
import time
import json
import hashlib
import ast

from mcp.server.fastmcp import FastMCP

from nbdev_mcp.utils.paths import (
    resolve_selector, iter_notebooks, modidx_path, settings_dict, read_nb, tutorials_dir
)
from nbdev_mcp.utils.nb import join_source
from nbdev_mcp.utils.depgraph import (
    build_graph, DependencyGraph, SymbolNode, Edge
)


## Data Structures

In [ ]:
#| export
@dataclass
class SymbolStats:
    """Usage statistics for a symbol.
    
    Attributes
    ----------
    fqn : str
        Fully qualified name (module.symbol).
    import_count : int
        Number of times this symbol is imported.
    call_count : int
        Estimated direct call count.
    subclass_count : int
        Number of classes inheriting (for classes).
    last_modified : float
        Last modification timestamp.
    notebooks_using : Set[str]
        Set of notebook IDs that use this symbol.
    kind : str
        Symbol type: function, class, constant, type.
    line_count : int
        Number of lines of code.
    """
    fqn: str
    import_count: int = 0
    call_count: int = 0
    subclass_count: int = 0
    last_modified: float = 0.0
    notebooks_using: Set[str] = field(default_factory=set)
    kind: str = 'function'
    line_count: int = 0
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert to JSON-serializable dict."""
        return {
            'fqn': self.fqn,
            'import_count': self.import_count,
            'call_count': self.call_count,
            'subclass_count': self.subclass_count,
            'last_modified': self.last_modified,
            'notebooks_using': list(self.notebooks_using),
            'kind': self.kind,
            'line_count': self.line_count
        }
    
    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> 'SymbolStats':
        """Create from dict."""
        return cls(
            fqn=d['fqn'],
            import_count=d.get('import_count', 0),
            call_count=d.get('call_count', 0),
            subclass_count=d.get('subclass_count', 0),
            last_modified=d.get('last_modified', 0.0),
            notebooks_using=set(d.get('notebooks_using', [])),
            kind=d.get('kind', 'function'),
            line_count=d.get('line_count', 0)
        )

In [ ]:
#| export
@dataclass
class ModuleStats:
    """Aggregated stats for a module.
    
    Attributes
    ----------
    module : str
        Module name.
    symbol_count : int
        Number of symbols in module.
    total_imports : int
        Total import count for all symbols.
    total_lines : int
        Total lines of code.
    complexity_score : float
        Approximate complexity score.
    """
    module: str
    symbol_count: int = 0
    total_imports: int = 0
    total_lines: int = 0
    complexity_score: float = 0.0
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert to JSON-serializable dict."""
        return asdict(self)
    
    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> 'ModuleStats':
        """Create from dict."""
        return cls(**d)

In [ ]:
#| export
@dataclass
class LiveModidxCache:
    """Cached _modidx with usage tallies.
    
    Attributes
    ----------
    project : Path
        Project root path.
    lib_name : str
        Library name.
    symbol_stats : Dict[str, SymbolStats]
        Stats per symbol FQN.
    module_stats : Dict[str, ModuleStats]
        Stats per module.
    dependency_graph : DependencyGraph
        Full dependency graph.
    build_time : float
        When cache was built.
    valid_until : float
        TTL-based expiry timestamp.
    modidx_mtime : float
        mtime of _modidx.py when cache was built.
    notebooks_mtime : float
        Max mtime of all notebooks when cache was built.
    """
    project: Path
    lib_name: str
    symbol_stats: Dict[str, SymbolStats] = field(default_factory=dict)
    module_stats: Dict[str, ModuleStats] = field(default_factory=dict)
    dependency_graph: Optional[DependencyGraph] = None
    build_time: float = 0.0
    valid_until: float = 0.0
    modidx_mtime: float = 0.0
    notebooks_mtime: float = 0.0
    
    def is_valid(self) -> bool:
        """Check if cache is still valid."""
        return time.time() < self.valid_until
    
    def is_stale(self) -> bool:
        """Check if source files have changed since cache was built."""
        try:
            # Check _modidx.py mtime
            midx_path = modidx_path(self.project)
            if midx_path.exists():
                if midx_path.stat().st_mtime > self.modidx_mtime:
                    return True
            
            # Check max notebook mtime
            current_max = 0.0
            for nb in iter_notebooks(self.project):
                current_max = max(current_max, nb.stat().st_mtime)
            if current_max > self.notebooks_mtime:
                return True
            
            return False
        except Exception:
            return True  # Assume stale on errors

## Cache Persistence

In [ ]:
#| export
CACHE_SNAPSHOT_DIR = Path.home() / '.nbdev_mcp' / 'cache'


def _project_hash(project: Path) -> str:
    """Generate a hash for the project path."""
    return hashlib.md5(str(project.resolve()).encode()).hexdigest()[:16]


def _snapshot_path(project: Path) -> Path:
    """Get the snapshot file path for a project."""
    return CACHE_SNAPSHOT_DIR / f"{_project_hash(project)}.json"


def save_cache_snapshot(cache: LiveModidxCache) -> Path:
    """Save cache to JSON snapshot.
    
    Parameters
    ----------
    cache : LiveModidxCache
        Cache to save.
    
    Returns
    -------
    Path
        Path to snapshot file.
    """
    CACHE_SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
    
    snapshot = {
        'project': str(cache.project),
        'lib_name': cache.lib_name,
        'build_time': cache.build_time,
        'modidx_mtime': cache.modidx_mtime,
        'notebooks_mtime': cache.notebooks_mtime,
        'symbol_stats': {k: v.to_dict() for k, v in cache.symbol_stats.items()},
        'module_stats': {k: v.to_dict() for k, v in cache.module_stats.items()}
    }
    
    out_path = _snapshot_path(cache.project)
    out_path.write_text(json.dumps(snapshot, indent=2), encoding='utf-8')
    return out_path


def load_cache_snapshot(project: Path, ttl_seconds: int = 300) -> Optional[LiveModidxCache]:
    """Load cache from JSON snapshot if valid.
    
    Parameters
    ----------
    project : Path
        Project path.
    ttl_seconds : int
        TTL for cache validity.
    
    Returns
    -------
    Optional[LiveModidxCache]
        Cache if snapshot exists and is not stale, else None.
    """
    snapshot_file = _snapshot_path(project)
    if not snapshot_file.exists():
        return None
    
    try:
        data = json.loads(snapshot_file.read_text(encoding='utf-8'))
        
        cache = LiveModidxCache(
            project=Path(data['project']),
            lib_name=data['lib_name'],
            build_time=data['build_time'],
            valid_until=time.time() + ttl_seconds,
            modidx_mtime=data['modidx_mtime'],
            notebooks_mtime=data['notebooks_mtime'],
            symbol_stats={k: SymbolStats.from_dict(v) for k, v in data['symbol_stats'].items()},
            module_stats={k: ModuleStats.from_dict(v) for k, v in data['module_stats'].items()}
        )
        
        # Check if stale
        if cache.is_stale():
            return None
        
        return cache
    except Exception:
        return None


def cleanup_old_snapshots(max_age_days: int = 7) -> int:
    """Remove old cache snapshots.
    
    Parameters
    ----------
    max_age_days : int
        Maximum age in days.
    
    Returns
    -------
    int
        Number of files removed.
    """
    if not CACHE_SNAPSHOT_DIR.exists():
        return 0
    
    cutoff = time.time() - (max_age_days * 24 * 60 * 60)
    removed = 0
    
    for f in CACHE_SNAPSHOT_DIR.glob('*.json'):
        if f.stat().st_mtime < cutoff:
            f.unlink()
            removed += 1
    
    return removed

## Cache Manager

In [ ]:
#| export
class ModidxCacheManager:
    """Manages live cache with automatic refresh.
    
    Singleton pattern - use get() to access cache.
    
    Examples
    --------
    >>> cache = ModidxCacheManager.get(project_path)
    >>> print(cache.symbol_stats['utils.config.get_config'].import_count)
    """
    
    _caches: ClassVar[Dict[str, LiveModidxCache]] = {}
    TTL_SECONDS: ClassVar[int] = 300  # 5 minutes
    
    @classmethod
    def get(
        cls,
        project: Path,
        force_refresh: bool = False,
        include_graph: bool = True
    ) -> LiveModidxCache:
        """Get or build cache for project.
        
        Parameters
        ----------
        project : Path
            Project root path.
        force_refresh : bool
            Force rebuild even if cache is valid.
        include_graph : bool
            Whether to include full dependency graph.
        
        Returns
        -------
        LiveModidxCache
            The cached data.
        """
        key = str(project.resolve())
        cache = cls._caches.get(key)
        
        # Check memory cache first
        if cache and not force_refresh and cache.is_valid() and not cache.is_stale():
            return cache
        
        # Try loading from disk snapshot
        if not force_refresh:
            disk_cache = load_cache_snapshot(project, cls.TTL_SECONDS)
            if disk_cache:
                # Rebuild graph if needed
                if include_graph:
                    disk_cache.dependency_graph = build_graph(str(project))
                cls._caches[key] = disk_cache
                return disk_cache
        
        # Build new cache
        cache = cls._build_cache(project, include_graph)
        cls._caches[key] = cache
        
        # Save to disk
        try:
            save_cache_snapshot(cache)
        except Exception:
            pass  # Disk save failed, continue with memory cache
        
        return cache
    
    @classmethod
    def _build_cache(cls, project: Path, include_graph: bool = True) -> LiveModidxCache:
        """Build complete cache with usage tallies."""
        settings = settings_dict(project)
        lib_name = settings.get('lib_name', 'pkg')
        
        # Get file mtimes for staleness detection
        modidx_mtime = 0.0
        midx_path = modidx_path(project)
        if midx_path.exists():
            modidx_mtime = midx_path.stat().st_mtime
        
        notebooks_mtime = 0.0
        for nb in iter_notebooks(project):
            notebooks_mtime = max(notebooks_mtime, nb.stat().st_mtime)
        
        # Build dependency graph
        graph = build_graph(str(project)) if include_graph else None
        
        # Compute symbol stats from graph
        symbol_stats = cls._compute_symbol_stats(graph) if graph else {}
        
        # Aggregate module stats
        module_stats = cls._compute_module_stats(symbol_stats)
        
        return LiveModidxCache(
            project=project,
            lib_name=lib_name,
            symbol_stats=symbol_stats,
            module_stats=module_stats,
            dependency_graph=graph,
            build_time=time.time(),
            valid_until=time.time() + cls.TTL_SECONDS,
            modidx_mtime=modidx_mtime,
            notebooks_mtime=notebooks_mtime
        )
    
    @classmethod
    def _compute_symbol_stats(
        cls,
        graph: DependencyGraph
    ) -> Dict[str, SymbolStats]:
        """Compute usage stats from dependency graph edges."""
        stats: Dict[str, SymbolStats] = {}
        
        # Initialize stats for all symbols
        for sym_id, sym in graph.symbols.items():
            stats[sym_id] = SymbolStats(
                fqn=sym_id,
                kind=sym.kind,
                line_count=sym.line_count
            )
        
        # Count imports from edges
        for edge in graph.edges:
            if edge.kind == 'import' and edge.source in stats:
                stats[edge.source].import_count += 1
                
                # Track which notebooks use this symbol
                target_nb = edge.target
                if target_nb in graph.symbols:
                    target_nb = graph.symbols[target_nb].notebook
                stats[edge.source].notebooks_using.add(target_nb)
        
        return stats
    
    @classmethod
    def _compute_module_stats(
        cls,
        symbol_stats: Dict[str, SymbolStats]
    ) -> Dict[str, ModuleStats]:
        """Aggregate symbol stats into module stats."""
        module_agg: Dict[str, Dict[str, Any]] = defaultdict(lambda: {
            'symbol_count': 0,
            'total_imports': 0,
            'total_lines': 0
        })
        
        for sym_id, sym_stats in symbol_stats.items():
            # Extract module from FQN (e.g., 'utils.config.get_config' -> 'utils.config')
            parts = sym_id.rsplit('.', 1)
            module = parts[0] if len(parts) > 1 else '_root'
            
            module_agg[module]['symbol_count'] += 1
            module_agg[module]['total_imports'] += sym_stats.import_count
            module_agg[module]['total_lines'] += sym_stats.line_count
        
        return {
            mod: ModuleStats(
                module=mod,
                symbol_count=agg['symbol_count'],
                total_imports=agg['total_imports'],
                total_lines=agg['total_lines'],
                complexity_score=agg['total_lines'] / max(1, agg['symbol_count'])
            )
            for mod, agg in module_agg.items()
        }
    
    @classmethod
    def invalidate(cls, project: Optional[Path] = None) -> None:
        """Invalidate cache (after exports, edits, etc.).
        
        Parameters
        ----------
        project : Path, optional
            Specific project to invalidate, or all if None.
        """
        if project:
            key = str(project.resolve())
            cls._caches.pop(key, None)
        else:
            cls._caches.clear()
    
    @classmethod
    def stats(cls) -> Dict[str, Any]:
        """Get cache manager statistics."""
        return {
            'cached_projects': len(cls._caches),
            'ttl_seconds': cls.TTL_SECONDS,
            'projects': [
                {
                    'path': str(c.project),
                    'symbols': len(c.symbol_stats),
                    'modules': len(c.module_stats),
                    'valid': c.is_valid(),
                    'stale': c.is_stale()
                }
                for c in cls._caches.values()
            ]
        }

## Cache-Aware Tools

In [ ]:
#| export
def get_symbol_usage(
    project: Optional[str] = None,
    symbol: Optional[str] = None
) -> Dict[str, Any]:
    """Get usage stats for a symbol from cache.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    symbol : str, optional
        Symbol FQN (e.g., 'utils.config.get_config').
    
    Returns
    -------
    Dict[str, Any]
        Symbol usage stats.
    """
    p = resolve_selector(project)
    cache = ModidxCacheManager.get(p)
    stats = cache.symbol_stats.get(symbol)
    
    if not stats:
        return {'ok': False, 'error': f'Symbol not found: {symbol}'}
    
    return {
        'ok': True,
        'stats': stats.to_dict()
    }


def get_hot_symbols(
    project: Optional[str] = None,
    top_n: int = 20
) -> Dict[str, Any]:
    """Get most-used symbols by import count.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    top_n : int
        Number of symbols to return.
    
    Returns
    -------
    Dict[str, Any]
        Top symbols by usage.
    """
    p = resolve_selector(project)
    cache = ModidxCacheManager.get(p)
    
    sorted_stats = sorted(
        cache.symbol_stats.values(),
        key=lambda s: s.import_count,
        reverse=True
    )[:top_n]
    
    return {
        'ok': True,
        'symbols': [s.to_dict() for s in sorted_stats]
    }


def get_orphan_symbols(
    project: Optional[str] = None,
    min_lines: int = 3
) -> Dict[str, Any]:
    """Get symbols with zero imports (candidates for removal).
    
    Orphans are not always bad (they can be used in tutorials/docs), but
    they can signal duplication when a new symbol is introduced instead.
    
    Notes
    -----
    Dead-code analysis is weighted by module hierarchy and tutorial usage.
    Lower-level (deeper) modules are more concerning, while tutorial usage
    suggests public API and lowers concern.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    min_lines : int
        Minimum line count to consider (skip trivial constants).
    
    Returns
    -------
    Dict[str, Any]
        Orphan symbols.
    """
    p = resolve_selector(project)
    cache = ModidxCacheManager.get(p)
    
    # Scan tutorials for symbol usage (public API signals)
    tuts = tutorials_dir(p)
    tutorial_symbols: set = set()
    if tuts.exists():
        for nb in tuts.rglob('*.ipynb'):
            data = read_nb(nb)
            for cell in data.get('cells', []):
                if cell.get('cell_type') != 'code':
                    continue
                src = join_source(cell.get('source', []))
                if not src.strip():
                    continue
                try:
                    tree = ast.parse(src)
                except SyntaxError:
                    continue
                for node in ast.walk(tree):
                    if isinstance(node, ast.Name):
                        tutorial_symbols.add(node.id)
                    elif isinstance(node, ast.Attribute):
                        tutorial_symbols.add(node.attr)
                    elif isinstance(node, ast.ImportFrom):
                        for alias in node.names:
                            if alias.name != '*':
                                tutorial_symbols.add(alias.name)
    
    orphans_raw = [
        s for s in cache.symbol_stats.values()
        if s.import_count == 0 and s.line_count >= min_lines
    ]
    
    orphans: List[Dict[str, Any]] = []
    for s in orphans_raw:
        if not s.fqn:
            continue
        module = s.fqn.rsplit('.', 1)[0] if '.' in s.fqn else ''
        name = s.fqn.split('.')[-1]
        module_depth = len([part for part in module.split('.') if part]) if module else 0
        used_in_tutorials = name in tutorial_symbols
        concern_weight = module_depth
        if used_in_tutorials:
            concern_weight = max(0, concern_weight - 1)
        entry = s.to_dict()
        entry.update({
            'module': module,
            'symbol': name,
            'module_depth': module_depth,
            'used_in_tutorials': used_in_tutorials,
            'concern_weight': concern_weight,
        })
        orphans.append(entry)
    
    # Sort by weight, then line count
    orphans.sort(key=lambda s: (s['concern_weight'], s['line_count']), reverse=True)
    
    note = (
        "Orphan symbols are not always bad (tutorials/docs), but they can signal duplication. "
        "Weights: deeper modules are more concerning; tutorial usage lowers concern."
    )
    
    return {
        'ok': True,
        'orphans': orphans,
        'total_lines': sum(s['line_count'] for s in orphans),
        'note': note,
        'weighting': {
            'tutorials_dir': str(tuts),
            'rule': 'concern_weight = module_depth; subtract 1 if used in tutorials'
        }
    }


def get_module_summary(
    project: Optional[str] = None
) -> Dict[str, Any]:
    """Get summary stats for all modules.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    
    Returns
    -------
    Dict[str, Any]
        Module statistics.
    """
    p = resolve_selector(project)
    cache = ModidxCacheManager.get(p)
    
    modules = sorted(
        cache.module_stats.values(),
        key=lambda m: m.total_imports,
        reverse=True
    )
    
    return {
        'ok': True,
        'modules': [m.to_dict() for m in modules],
        'total_symbols': sum(m.symbol_count for m in modules),
        'total_lines': sum(m.total_lines for m in modules)
    }


## MCP Tool Registration

In [ ]:
#| export
def add_cache_tools(mcp: FastMCP) -> None:
    """Register cache-aware tools with MCP server.
    
    Parameters
    ----------
    mcp : FastMCP
        The MCP server instance.
    """
    
    @mcp.tool()
    def symbol_usage(
        project: Optional[str] = None,
        symbol: str = ''
    ) -> Dict[str, Any]:
        """Get usage statistics for a symbol.
        
        Shows import count, which notebooks use it, and line count.
        Uses cached data for fast response.
        """
        return get_symbol_usage(project, symbol)
    
    @mcp.tool()
    def hot_symbols(
        project: Optional[str] = None,
        top_n: int = 20
    ) -> Dict[str, Any]:
        """Get most frequently imported symbols.
        
        Useful for identifying core utilities and potential
        candidates for optimization.
        """
        return get_hot_symbols(project, top_n)
    
    @mcp.tool()
    def orphan_symbols(
        project: Optional[str] = None,
        min_lines: int = 3
    ) -> Dict[str, Any]:
        """Find symbols with zero imports (dead code candidates).
        
        Excludes trivial constants. Useful for cleanup. Orphans are not
        always bad (tutorials/docs), but can indicate duplication.
        """
        return get_orphan_symbols(project, min_lines)
    
    @mcp.tool()
    def module_summary(
        project: Optional[str] = None
    ) -> Dict[str, Any]:
        """Get summary statistics for all modules.
        
        Shows symbol counts, import totals, and complexity.
        """
        return get_module_summary(project)
    
    @mcp.tool()
    def refresh_cache(
        project: Optional[str] = None
    ) -> Dict[str, Any]:
        """Force refresh the symbol cache.
        
        Use after significant changes to notebooks or exports.
        """
        p = resolve_selector(project)
        ModidxCacheManager.invalidate(p)
        cache = ModidxCacheManager.get(p, force_refresh=True)
        
        return {
            'ok': True,
            'message': 'Cache refreshed',
            'symbols': len(cache.symbol_stats),
            'modules': len(cache.module_stats)
        }
    
    @mcp.tool()
    def cache_status() -> Dict[str, Any]:
        """Get cache manager status.
        
        Shows cached projects, validity, and staleness.
        """
        return {
            'ok': True,
            **ModidxCacheManager.stats()
        }

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()